# Text Cleaning for NLP News Project

This notebook cleans raw news article text for downstream topic modeling, named entity recognition, and sentiment analysis.

Main tasks:
- load the dataset
- inspect raw article structure
- remove boilerplate / UI noise
- extract main article content
- save cleaned text for later notebooks

In [ ]:
import pandas as pd
import re

pd.set_option("display.max_colwidth", None)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4


In [ ]:
df_news_final_project = pd.read_parquet('https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet', engine='pyarrow')
df_news_final_project.shape



(199989, 5)

In [ ]:
df_news_final_project['date'].sort_values(ascending=False).tail(5)

,date
182525,2022-01-01
2199,2022-01-01
185147,2022-01-01
117462,2022-01-01
94772,2022-01-01


In [ ]:
df_news_final_project['date'] = pd.to_datetime(df_news_final_project['date'])
df_news_final_project['date'].dtype

dtype('<M8[ns]')

In [ ]:
df_news_final_project['text'].sample(10, random_state=22)

181051    Sales Compensation Analyst AI Agent | ClickUp™The everything  app, for work.ProductSolutionsResourcesPricingEnterpriseContact SalesLog inSign UpSales Compensation Analyst AI AgentDiscover how AI Agents can transform your workflow, boost productivity, and help you achieve more with less.Start Today for FreeBoost your team's efficiency and precision with AI Agents for Sales Compensation Analysis! These intelligent tools automate tedious data tasks, eliminate errors, and allow you to focus on strategic insights rather than number crunching. ClickUp Brain enhances this transformation by centralizing your data-driven decisions, making every sales compensation strategy smarter and faster.AI Agents for Sales Compensation AnalystsSales Compensation Analyst AI Agents are like the ultimate sidekick for sales teams. They seamlessly handle complex compensation plans, ensuring that everyone gets rewarded fairly and promptly. These digital agents are designed to reduce the time spent on administrative tasks, so your team can focus on their main goal—driving sales.Types of AI Agents & Their Roles:Data Validation Agent: Checks the accuracy of sales data to ensure compensation calculations are correct.Incentive Plan Agent: Designs and updates compensation plans based on sales targets and performance.Payout Calculation Agent: Automates the process of calculating and distributing sales commissions efficiently.Performance Tracking Agent: Monitors sales performance metrics to provide insights into team achievements and areas for improvement.AI agents in sales compensation environments primarily focus on automating and streamlining processes. For example, a Payout Calculation Agent will take the month's sales data, apply specific rules and formulas, and calculate each team member's compensation. This minimizes errors and ensures everyone is paid correctly based on their goals and achievements. Performance Tracking Agents can highlight top performers and identify trends, helping managers make informed decisions about motivating and rewarding their teams. With AI agents on your side, keeping track of complex compensation structures becomes less of a chore and more of an opportunity to enhance team performance.Benefits of Using AI Agents for Sales Compensation AnalystsAI Agents are quickly transforming how Sales Compensation Analysts operate, making everyday tasks more efficient and strategic decisions smarter. Here's why implementing AI Agents can revolutionize your sales compensation processes:Increased EfficiencyTime-Saving Automation: Automate repetitive tasks like data entry and report generation, freeing up analysts to focus on strategic activities.Rapid Data Processing: Analyze huge volumes of compensation data in seconds, ensuring timely adjustments and payouts.Enhanced AccuracyError Reduction: Mitigate human error in calculations and data handling, ensuring everyone gets paid accurately.Consistent Updates: Maintain up-to-date records with consistent, real-time data verification and corrections.Improved Decision MakingAdvanced Analytics: Gain insights through predictive analytics, helping to forecast sales trends and make informed compensation decisions.Scenario Simulation: Test different compensation models and their potential impact without any real-world consequences.Scalable SolutionsEffortless Scaling: Adapt seamlessly to the growth of your sales teams and complexity of compensation plans without extra manpower.Customizable Framework: Tailor AI functions to suit complex and varied compensation models within a few clicks.Enhanced Employee SatisfactionTransparent Processes: Increase trust by maintaining transparent and fair compensation processes, boosting morale and motivation.Prompt Resolution: Rapidly address and resolve any payment discrepancies, which fosters a culture of reliability and trust.Incorporating AI Agents into your sales compensation strategy doesn't just streamline processes—it transforms them, fostering a more e

In [ ]:
END_MARKERS = [
    # strong footer blocks
    "DISCLAIMER FOR COMMENTS",
    "Available for Roku",
    "WFMZ+",
    "Lehigh Valley News",
    "Berks Area News",
    "Sections",
    "Services",
    "Advertising:",
    "This website is not intended for users located within the European Economic Area.",
    "Do Not Sell My",
    "Ad Choices",
    "Manage Preferences",

    # generic across sites
    "Trending Stories", "Latest Stories", "RELATED ARTICLES", "Leave a reply",
    "READ MORE ON:", "FIRST PUBLISHED IN:", "POST / READ COMMENTS",
    "TRENDING", "Latest News", "Connect us on", "Give Feedback",
    "Privacy Policy", "Terms of Use", "Public File", "DMCA",
    "All Rights Reserved", "Copyright", "©"
]

END_MARKERS += [
    "Open navigation",
    "Toggle",
    "Sponsored",
    "Enter term",
    "Search for:",
    "Skip to content",
    "Skip to main content",
    "Sign Up",
    "Log In",
    "Logout",
    "Profile",
    "Send Us A News Tip",
]

NOISE_PATTERNS = [
    r"\bcookie(s)?\b",
    r"\bgdpr\b",
    r"\bfeed opens?\b",
    r"\bsets cookie\b",
    r"\bnewswires?\b",
    r"\bindustry newswires?\b",
    r"\bopen navigation\b",
    r"\bvisitor agreement\b",
    r"\bmarket segmentation\b"
]

def remove_noise_phrases(text):
    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

def find_prose_start(s: str) -> int:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    for m in re.finditer(r"[^\n]+", s):
        line = m.group(0).strip()
        if len(line) >= 160 and re.search(r"[.!?]", line) and sum(c.isalpha() for c in line) >= 100:
            return m.start()
    return 0

def extract_main(raw: str, title: str) -> str:
    s = raw.replace("\r\n", "\n").replace("\r", "\n").replace("\xa0", " ")

    start_idx = None

    # 1) Start: last occurrence of title
    if isinstance(title, str) and title.strip():
        t = re.escape(title.strip())
        matches = list(re.finditer(t, s, flags=re.IGNORECASE))
        if matches:
            start_idx = matches[-1].start()

    # 2) Better start anchor: PRNewswire dateline (works great for press releases)
    if start_idx is None:
        m = re.search(r"/PRNewswire/\s*--", s, flags=re.IGNORECASE)
        if m:
            # start a bit before to keep lead sentence
            start_idx = max(0, m.start() - 120)

    # 3) Fallback: Published/Updated
    if start_idx is None:
        m = re.search(r"\b(Published|Updated)\b", s, flags=re.IGNORECASE)
        if m:
            start_idx = m.start()

    # 4) Fallback: first prose paragraph
    if start_idx is None:
        start_idx = find_prose_start(s)

    s2 = s[start_idx:]

    # -------------- END CUT (robust) --------------
    # Use BOTH string markers and regex patterns (handles spacing/formatting variance)
    lower = s2.lower()
    cut = None

    for marker in END_MARKERS:
        idx = lower.find(marker.lower())
        if idx != -1:
            cut = idx if cut is None else min(cut, idx)

    # Regex fallback for disclaimer blocks
    m = re.search(r"\bdisclaimer\s+for\s+comments\b", s2, flags=re.IGNORECASE)
    if m:
        cut = m.start() if cut is None else min(cut, m.start())

    if cut is not None and cut > 300:
        s2 = s2[:cut]

    # Normalize whitespace and remove URLs
    s2 = re.sub(r"http\S+|www\.\S+", " ", s2)
    s2 = re.sub(r"\s+", " ", s2).strip()
    return s2

UI_PHRASES = [
    "View Less", "View More", "Show Search",
    "Dashboard", "My Account", "Site search", "Search",
    "Print", "Save", "Share This",                 # WFMZ UI
    "Facebook", "Twitter", "WhatsApp", "SMS", "Email"
]
UI_PATTERNS = [
    r"\bopen\s+navigation\b",
    r"\btoggle\b",
    r"\bsponsored\b",
    r"\benter\s+term\b",
    r"\bskip\s+to\s+(main\s+)?content\b",
    r"\bsign\s*up\b",
    r"\blog\s*in\b",
    r"\blog\s*out\b",
    r"\bmy\s+account\b",
    r"\bsite\s+search\b",
    r"\bsearch\b",
    r"\bshare\b",
    r"\bprint\b",
    r"\bsave\b",
    r"\bfacebook\b|\btwitter\b|\bwhatsapp\b|\bsms\b|\bemail\b",
]

def strip_publish_update_meta(text: str) -> str:
    # Published <Month> <D>, <YYYY> (at|@) <H:MM> AM/PM
    text = re.sub(
        r"\bPublished\b[:\s]*[A-Za-z]{3,9}\s+\d{1,2},\s+\d{4}\s*(?:at|@)?\s*\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )
    # Updated <Month> <D>, <YYYY> (at|@) <H:MM> AM/PM   (optional "Full Forecast")
    text = re.sub(
        r"\bUpdated\b[:\s]*[A-Za-z]{3,9}\s+\d{1,2},\s+\d{4}\s*(?:at|@)?\s*\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)\b(?:\s*Full\s*Forecast)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Also remove stray tokens that sometimes remain
    text = re.sub(r"\bFull\s*Forecast\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def light_cleanup(text: str) -> str:
    for pat in UI_PATTERNS:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    # common junk tokens that show up as “enter term” fragments
    text = re.sub(r"\b(term|toggle|navigation)\b", " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_text_final(raw, title):
    main = extract_main(raw, title)
    main = light_cleanup(main)
    main = strip_weather_updated(main)
    main = remove_noise_phrases(main)
    return main

In [ ]:
df_news_final_project["clean_text"] = df_news_final_project.apply(lambda r: clean_text_final(r["text"], r["title"]), axis=1)

# basic filters
df_news_final_project["clean_len"] = df_news_final_project["clean_text"].str.len()
df_news_final_project = df_news_final_project[df_news_final_project["clean_len"].fillna(0) >= 300].copy()   # drop ultra-short junk
df_news_final_project = df_news_final_project.drop_duplicates(subset=["url"])            # if url unique

In [ ]:
project_cleaned = df_news_final_project[["url", "date", "title", "clean_text"]].copy()

project_cleaned.to_parquet(
    "/content/drive/MyDrive/NLP Final Project/news_cleaned.parquet",
    index=False
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

df = pd.read_parquet(
    "/content/drive/MyDrive/NLP Final Project/docs_with_topics.parquet"
)

Mounted at /content/drive
